Data Collection 2 (Scraping)

Source scraped: English Wikipedia artist articles (one per top-10 finisher).

Summary:
- Scraping tool: requests + BeautifulSoup.
- Around 99 unique artist Wikipedia pages.
- For each artist we extract the infobox (the box on the top-right of every artist page) — fields like Born, Origin, Genres, Years active, Labels, Occupation.
- Output joins with 01_api_collection.ipynb on the artist column. This gives us, per artist: their full release history (from the API) plus background context like "when did they start their career?" which is essential for measuring a career bump


Ethics & legality note:

- robots.txt check. https://en.wikipedia.org/robots.txt allows generic crawlers on /wiki/ article paths.
- Terms of Service. Wikipedia text is CC-BY-SA 3.0. We will attribute "Wikipedia contributors" in the final report and cite the URLs used.
- No personal data. Artists are public figures performing under stage names. No emails, phone numbers, home addresses, or private photos. No GDPR concerns.
- Rate-limit choice. 99 requests with time.sleep(1) between them and timeout=15 per request equals to roughly 2 minutes total. Well under any reasonable abuse threshold.
- Identification. Descriptive User-Agent with a contact email, as Wikipedia recommends.


In [1]:
!pip install -q pyarrow

In [2]:
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

HEADERS = {
    "User-Agent": "AlbertSchool-DataProject/1.0 (marta.prandin@student.example.com)",
    "Accept": "text/html",
}
BASE = "https://en.wikipedia.org/wiki/"

The 100 top-10 finishers with their Wikipedia URLs

Each row is (artist, year, song, country, rank, wikipedia_slug). The slug is the URL-path part after /wiki/. For artists with common names we use the disambiguated form (e.g. JJ_(Austrian_singer)).

In [3]:
TOP10 = [
    ("Måns Zelmerlöw", 2015, "Heroes", "Sweden", 1, "Måns_Zelmerlöw"),
    ("Polina Gagarina", 2015, "A Million Voices", "Russia", 2, "Polina_Gagarina"),
    ("Il Volo", 2015, "Grande amore", "Italy", 3, "Il_Volo"),
    ("Loïc Nottet", 2015, "Rhythm Inside", "Belgium", 4, "Loïc_Nottet"),
    ("Aminata Savadogo", 2015, "Love Injected", "Latvia", 5, "Aminata_Savadogo"),
    ("Guy Sebastian", 2015, "Tonight Again", "Australia", 6, "Guy_Sebastian"),
    ("Elina Born and Stig Rästa", 2015, "Goodbye to Yesterday", "Estonia", 7, "Elina_Born"),
    ("Mørland", 2015, "A Monster Like Me", "Norway", 8, "Mørland_(singer)"),
    ("Bojana Stamenov", 2015, "Beauty Never Lies", "Serbia", 9, "Bojana_Stamenov"),
    ("Nadav Guedj", 2015, "Golden Boy", "Israel", 10, "Nadav_Guedj"),
    ("Jamala", 2016, "1944", "Ukraine", 1, "Jamala"),
    ("Dami Im", 2016, "Sound of Silence", "Australia", 2, "Dami_Im"),
    ("Sergey Lazarev", 2016, "You Are the Only One", "Russia", 3, "Sergey_Lazarev"),
    ("Poli Genova", 2016, "If Love Was a Crime", "Bulgaria", 4, "Poli_Genova"),
    ("Frans", 2016, "If I Were Sorry", "Sweden", 5, "Frans_(singer)"),
    ("Amir", 2016, "J'ai cherché", "France", 6, "Amir_(French_singer)"),
    ("Iveta Mukuchyan", 2016, "LoveWave", "Armenia", 7, "Iveta_Mukuchyan"),
    ("Michał Szpak", 2016, "Color of Your Life", "Poland", 8, "Michał_Szpak"),
    ("Donny Montell", 2016, "I've Been Waiting for This Night", "Lithuania", 9, "Donny_Montell"),
    ("Laura Tesoro", 2016, "What's the Pressure", "Belgium", 10, "Laura_Tesoro"),
    ("Salvador Sobral", 2017, "Amar pelos dois", "Portugal", 1, "Salvador_Sobral"),
    ("Kristian Kostov", 2017, "Beautiful Mess", "Bulgaria", 2, "Kristian_Kostov"),
    ("SunStroke Project", 2017, "Hey, Mamma!", "Moldova", 3, "SunStroke_Project"),
    ("Blanche", 2017, "City Lights", "Belgium", 4, "Blanche_(singer)"),
    ("Robin Bengtsson", 2017, "I Can't Go On", "Sweden", 5, "Robin_Bengtsson"),
    ("Francesco Gabbani", 2017, "Occidentali's Karma", "Italy", 6, "Francesco_Gabbani"),
    ("Ilinca and Alex Florea", 2017, "Yodel It!", "Romania", 7, "Ilinca_Băcilă"),
    ("Joci Pápai", 2017, "Origo", "Hungary", 8, "Joci_Pápai"),
    ("Jacques Houdek", 2017, "My Friend", "Croatia", 9, "Jacques_Houdek"),
    ("JOWST", 2017, "Grab the Moment", "Norway", 10, "JOWST"),
    ("Netta", 2018, "Toy", "Israel", 1, "Netta_Barzilai"),
    ("Eleni Foureira", 2018, "Fuego", "Cyprus", 2, "Eleni_Foureira"),
    ("Cesár Sampson", 2018, "Nobody but You", "Austria", 3, "Cesár_Sampson"),
    ("Michael Schulte", 2018, "You Let Me Walk Alone", "Germany", 4, "Michael_Schulte_(singer)"),
    ("Ermal Meta and Fabrizio Moro", 2018, "Non mi avete fatto niente", "Italy", 5, "Ermal_Meta"),
    ("Mikolas Josef", 2018, "Lie to Me", "Czech Republic", 6, "Mikolas_Josef"),
    ("Benjamin Ingrosso", 2018, "Dance You Off", "Sweden", 7, "Benjamin_Ingrosso"),
    ("Elina Nechayeva", 2018, "La forza", "Estonia", 8, "Elina_Nechayeva"),
    ("Rasmussen", 2018, "Higher Ground", "Denmark", 9, "Rasmussen_(singer)"),
    ("DoReDoS", 2018, "My Lucky Day", "Moldova", 10, "DoReDoS"),
    ("Duncan Laurence", 2019, "Arcade", "Netherlands", 1, "Duncan_Laurence"),
    ("Mahmood", 2019, "Soldi", "Italy", 2, "Mahmood_(singer)"),
    ("Sergey Lazarev", 2019, "Scream", "Russia", 3, "Sergey_Lazarev"),
    ("Luca Hänni", 2019, "She Got Me", "Switzerland", 4, "Luca_Hänni"),
    ("KEiiNO", 2019, "Spirit in the Sky", "Norway", 5, "KEiiNO"),
    ("John Lundvik", 2019, "Too Late for Love", "Sweden", 6, "John_Lundvik"),
    ("Chingiz", 2019, "Truth", "Azerbaijan", 7, "Chingiz_Mustafayev_(singer)"),
    ("Lake Malawi", 2019, "Friend of a Friend", "Czech Republic", 8, "Lake_Malawi_(band)"),
    ("Kate Miller-Heidke", 2019, "Zero Gravity", "Australia", 9, "Kate_Miller-Heidke"),
    ("Hatari", 2019, "Hatrið mun sigra", "Iceland", 10, "Hatari_(band)"),
    ("Måneskin", 2021, "Zitti e buoni", "Italy", 1, "Måneskin"),
    ("Barbara Pravi", 2021, "Voilà", "France", 2, "Barbara_Pravi"),
    ("Gjon's Tears", 2021, "Tout l'univers", "Switzerland", 3, "Gjon's_Tears"),
    ("Daði og Gagnamagnið", 2021, "10 Years", "Iceland", 4, "Daði_og_Gagnamagnið"),
    ("Go_A", 2021, "Shum", "Ukraine", 5, "Go_A"),
    ("Blind Channel", 2021, "Dark Side", "Finland", 6, "Blind_Channel"),
    ("Destiny", 2021, "Je Me Casse", "Malta", 7, "Destiny_Chukunyere"),
    ("The Roop", 2021, "Discoteque", "Lithuania", 8, "The_Roop"),
    ("Manizha", 2021, "Russian Woman", "Russia", 9, "Manizha"),
    ("The Black Mamba", 2021, "Love Is on My Side", "Portugal", 10, "The_Black_Mamba_(band)"),
    ("Kalush Orchestra", 2022, "Stefania", "Ukraine", 1, "Kalush_Orchestra"),
    ("Sam Ryder", 2022, "Space Man", "United Kingdom", 2, "Sam_Ryder"),
    ("Chanel", 2022, "SloMo", "Spain", 3, "Chanel_(singer)"),
    ("Cornelia Jakobs", 2022, "Hold Me Closer", "Sweden", 4, "Cornelia_Jakobs"),
    ("Konstrakta", 2022, "In corpore sano", "Serbia", 5, "Konstrakta"),
    ("Mahmood and Blanco", 2022, "Brividi", "Italy", 6, "Mahmood_(singer)"),
    ("Zdob și Zdub and Frații Advahov", 2022, "Trenulețul", "Moldova", 7, "Zdob_și_Zdub"),
    ("Amanda Tenfjord", 2022, "Die Together", "Greece", 8, "Amanda_Tenfjord"),
    ("MARO", 2022, "Saudade, saudade", "Portugal", 9, "Maro_(singer)"),
    ("Subwoolfer", 2022, "Give That Wolf a Banana", "Norway", 10, "Subwoolfer"),
    ("Loreen", 2023, "Tattoo", "Sweden", 1, "Loreen_(singer)"),
    ("Käärijä", 2023, "Cha Cha Cha", "Finland", 2, "Käärijä"),
    ("Noa Kirel", 2023, "Unicorn", "Israel", 3, "Noa_Kirel"),
    ("Marco Mengoni", 2023, "Due vite", "Italy", 4, "Marco_Mengoni"),
    ("Alessandra", 2023, "Queen of Kings", "Norway", 5, "Alessandra_Mele"),
    ("Tvorchi", 2023, "Heart of Steel", "Ukraine", 6, "Tvorchi"),
    ("Gustaph", 2023, "Because of You", "Belgium", 7, "Gustaph"),
    ("Alika", 2023, "Bridges", "Estonia", 8, "Alika_Milova"),
    ("Monika Linkytė", 2023, "Stay", "Lithuania", 9, "Monika_Linkytė"),
    ("Voyager", 2023, "Promise", "Australia", 10, "Voyager_(Australian_band)"),
    ("Nemo", 2024, "The Code", "Switzerland", 1, "Nemo_(singer)"),
    ("Baby Lasagna", 2024, "Rim Tim Tagi Dim", "Croatia", 2, "Baby_Lasagna"),
    ("Alyona Alyona and Jerry Heil", 2024, "Teresa & Maria", "Ukraine", 3, "Alyona_Alyona"),
    ("Slimane", 2024, "Mon amour", "France", 4, "Slimane_(singer)"),
    ("Eden Golan", 2024, "Hurricane", "Israel", 5, "Eden_Golan"),
    ("Bambie Thug", 2024, "Doomsday Blue", "Ireland", 6, "Bambie_Thug"),
    ("Angelina Mango", 2024, "La noia", "Italy", 7, "Angelina_Mango"),
    ("Ladaniva", 2024, "Jako", "Armenia", 8, "Ladaniva"),
    ("Marcus and Martinus", 2024, "Unforgettable", "Sweden", 9, "Marcus_&_Martinus"),
    ("Iolanda", 2024, "Grito", "Portugal", 10, "Iolanda_(singer)"),
    ("JJ", 2025, "Wasted Love", "Austria", 1, "JJ_(Austrian_singer)"),
    ("Yuval Raphael", 2025, "New Day Will Rise", "Israel", 2, "Yuval_Raphael"),
    ("Tommy Cash", 2025, "Espresso Macchiato", "Estonia", 3, "Tommy_Cash"),
    ("KAJ", 2025, "Bara bada bastu", "Sweden", 4, "KAJ_(group)"),
    ("Lucio Corsi", 2025, "Volevo essere un duro", "Italy", 5, "Lucio_Corsi"),
    ("Klavdia", 2025, "Asteromata", "Greece", 6, "Klavdia"),
    ("Louane", 2025, "Maman", "France", 7, "Louane"),
    ("Shkodra Elektronike", 2025, "Zjerm", "Albania", 8, "Shkodra_Elektronike"),
    ("Ziferblat", 2025, "Bird of Pray", "Ukraine", 9, "Ziferblat_(band)"),
    ("Zoë Më", 2025, "Voyage", "Switzerland", 10, "Zoë_Më"),
]

unique_slugs = list({row[5]: row for row in TOP10 if row[5]}.values())
print(f"Unique artists to scrape: {len(unique_slugs)}")

Unique artists to scrape: 98


Helper — scrape one artist's infobox:

Wikipedia artist pages use a `<table class="infobox vcard">`on the top-right. We extract every Label → Value row from it.
We also detect "wrong page" cases (e.g. the URL went to a disambiguation page) by checking whether the page text mentions Eurovision anywhere — if it doesn't, something is off and we skip it.

In [4]:
def scrape_artist(slug):

    url = BASE + slug
    response = requests.get(url, headers=HEADERS, timeout=15)
    if response.status_code == 404:
        return None
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")


    if "eurovision" not in response.text.lower():
        return {"_warning": "page does not mention Eurovision — possibly wrong URL"}


    infobox = soup.find("table", class_=lambda c: c and "infobox" in c)
    if infobox is None:
        return {"_warning": "no infobox found"}


    info = {}
    for row in infobox.find_all("tr"):
        th = row.find("th")
        td = row.find("td")
        if th and td:
            label = th.get_text(" ", strip=True)
            value = td.get_text(" ", strip=True)
            info[label] = value

    info["_source_url"] = url
    info["_scraped_at"] = datetime.now().isoformat(timespec="seconds")
    return info

Loop over every unique artist: 99 requests with a 1-second delay will roughly give 2 minutes.

In [5]:
records = []
errors = []

for row in unique_slugs:
    artist, year, song, country, rank, slug = row
    print(f"{artist} ({slug})")
    try:
        info = scrape_artist(slug)
    except requests.RequestException as e:
        print(f"  ! request error: {e}")
        errors.append({"artist": artist, "slug": slug, "error": str(e)})
        continue
    if info is None:
        print(f"  ! 404")
        errors.append({"artist": artist, "slug": slug, "error": "404"})
        continue
    if "_warning" in info:
        print(f"  ! {info['_warning']}")
        errors.append({"artist": artist, "slug": slug, "error": info["_warning"]})
        continue

    record = {
        "artist": artist,
        "earliest_eurovision_year": year,
        "earliest_eurovision_rank": rank,
        "wikipedia_slug": slug,
        "wikipedia_url": info.get("_source_url"),
        "scraped_at": info.get("_scraped_at"),
        "born": info.get("Born"),
        "origin": info.get("Origin"),
        "genres": info.get("Genres") or info.get("Genre"),
        "occupation": info.get("Occupation") or info.get("Occupations"),
        "years_active": info.get("Years active"),
        "labels": info.get("Labels") or info.get("Label"),
    }
    records.append(record)
    time.sleep(1)

print(f"\nDone. {len(records)} artist records collected.")
if errors:
    print(f"{len(errors)} artists could not be scraped:")
    for e in errors:
        print(f"  - {e['artist']} ({e['slug']}): {e['error']}")

Måns Zelmerlöw (Måns_Zelmerlöw)
Polina Gagarina (Polina_Gagarina)
Il Volo (Il_Volo)
Loïc Nottet (Loïc_Nottet)
Aminata Savadogo (Aminata_Savadogo)
Guy Sebastian (Guy_Sebastian)
Elina Born and Stig Rästa (Elina_Born)
Mørland (Mørland_(singer))
  ! 404
Bojana Stamenov (Bojana_Stamenov)
Nadav Guedj (Nadav_Guedj)
Jamala (Jamala)
Dami Im (Dami_Im)
Sergey Lazarev (Sergey_Lazarev)
Poli Genova (Poli_Genova)
Frans (Frans_(singer))
Amir (Amir_(French_singer))
  ! 404
Iveta Mukuchyan (Iveta_Mukuchyan)
Michał Szpak (Michał_Szpak)
Donny Montell (Donny_Montell)
Laura Tesoro (Laura_Tesoro)
Salvador Sobral (Salvador_Sobral)
Kristian Kostov (Kristian_Kostov)
SunStroke Project (SunStroke_Project)
Blanche (Blanche_(singer))
Robin Bengtsson (Robin_Bengtsson)
Francesco Gabbani (Francesco_Gabbani)
Ilinca and Alex Florea (Ilinca_Băcilă)
Joci Pápai (Joci_Pápai)
Jacques Houdek (Jacques_Houdek)
JOWST (JOWST)
Netta (Netta_Barzilai)
Eleni Foureira (Eleni_Foureira)
Cesár Sampson (Cesár_Sampson)
Michael Schulte (Mic

Build the DataFrame and inspect it

In [6]:
df = pd.DataFrame(records)
print("Shape:", df.shape)
print()
df.info()
print()
df.head()

Shape: (90, 12)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   artist                    90 non-null     object
 1   earliest_eurovision_year  90 non-null     int64 
 2   earliest_eurovision_rank  90 non-null     int64 
 3   wikipedia_slug            90 non-null     object
 4   wikipedia_url             90 non-null     object
 5   scraped_at                90 non-null     object
 6   born                      69 non-null     object
 7   origin                    42 non-null     object
 8   genres                    83 non-null     object
 9   occupation                66 non-null     object
 10  years_active              81 non-null     object
 11  labels                    58 non-null     object
dtypes: int64(2), object(10)
memory usage: 8.6+ KB



,artist,earliest_eurovision_year,earliest_eurovision_rank,wikipedia_slug,wikipedia_url,scraped_at,born,origin,genres,occupation,years_active,labels
0,Måns Zelmerlöw,2015,1,Måns_Zelmerlöw,https://en.wikipedia.org/wiki/Måns_Zelmerlöw,2026-05-20T09:29:24,Måns Petter Albert Sahlén Zelmerlöw ( 1986-06-...,None,Pop,Singer songwriter television presenter model a...,2005–present,Warner Music Sweden
1,Polina Gagarina,2015,2,Polina_Gagarina,https://en.wikipedia.org/wiki/Polina_Gagarina,2026-05-20T09:29:26,Polina Sergeyevna Gagarina ( 1987-03-27 ) 27 M...,None,pop,singer songwriter actress,2003–present,None
2,Il Volo,2015,3,Il_Volo,https://en.wikipedia.org/wiki/Il_Volo,2026-05-20T09:29:27,None,Italy,Operatic pop classical crossover,None,2010–present,Geffen Universal / Interscope Sony Music Latin
3,Loïc Nottet,2015,4,Loïc_Nottet,https://en.wikipedia.org/wiki/Loïc_Nottet,2026-05-20T09:29:29,Loïc Jean-Pierre Nottet ( 1996-04-10 ) 10 Apri...,None,Electropop synthpop [ 1 ] [ 2 ],Singer songwriter dancer,2014–present,Sony
4,Aminata Savadogo,2015,5,Aminata_Savadogo,https://en.wikipedia.org/wiki/Aminata_Savadogo,2026-05-20T09:29:31,Aminata Savadogo ( 1993-01-09 ) 9 January 1993...,None,Pop electronic R&B soul funk,Singer songwriter record producer model,2008–present,None


Save raw CSV and Parquet

In [7]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_name     = f"eurovision_artist_bios_{stamp}.csv"
parquet_name = f"eurovision_artist_bios_{stamp}.parquet"

df.to_csv(csv_name, index=False)
df.to_parquet(parquet_name, index=False)

print("Saved:", csv_name)
print("Saved:", parquet_name)

Saved: eurovision_artist_bios_20260520_093136.csv
Saved: eurovision_artist_bios_20260520_093136.parquet


Quick sanity check

In [8]:
print("Records:", len(df))
print()
print("Years active values (first 10):")
print(df["years_active"].dropna().head(10).tolist())
print()
print("Genres values (first 5):")
print(df["genres"].dropna().head(5).tolist())

Records: 90

Years active values (first 10):
['2005–present', '2003–present', '2010–present', '2014–present', '2008–present', '2012–present', '2009-present', '2015-present', '2001–present', '2005–present']

Genres values (first 5):
['Pop', 'pop', 'Operatic pop classical crossover', 'Electropop synthpop [ 1 ] [ 2 ]', 'Pop electronic R&B soul funk']


Download the files

In [11]:
from google.colab import files
files.download(csv_name)
files.download(parquet_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>